[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S09/S09_02_pca.ipynb)

# S09 · 02 — Reducir dimensiones con PCA

**Tiempo estimado:** 1 h · **Unidad del temario:** 3.10 (PCA, con mención a t-SNE)

**Contexto TurisData.** Dirección quiere *ver* a sus huéspedes en una sola imagen, pero
tenemos 7 variables por persona y no se puede dibujar en 7 dimensiones. **PCA** (análisis de
componentes principales) resume muchas variables en pocas "variables nuevas" (componentes) que
conservan la mayor parte de la información. Sirve para **visualizar**, para **quitar ruido** y
para **acelerar** otros modelos.

**Al terminar sabrás:**
1. Qué hace PCA con una analogía sencilla y cuándo usarlo.
2. Leer la varianza explicada y decidir cuántos componentes conservar.
3. Dibujar los huéspedes en 2D y colorearlos por segmento.
4. Interpretar las "cargas": qué variables componen cada componente.
5. Saber qué es t-SNE y cuándo *no* fiarse de sus gráficos.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

SEMILLA = 42
np.random.seed(SEMILLA)
sns.set_theme(style="whitegrid")

huespedes = pd.read_csv(dato("huespedes.csv"))
variables = [c for c in huespedes.columns if c != "id_huesped"]
X = StandardScaler().fit_transform(huespedes[variables])
grupo = KMeans(n_clusters=4, n_init=10, random_state=SEMILLA).fit_predict(X)
print(X.shape)

## 1. La idea de PCA

**Analogía.** Imagina una nube de puntos con forma de puro. Para describirla no necesitas los tres
ejes de la habitación: te basta **un eje a lo largo del puro** (donde hay más variación) y otro
más corto a lo ancho. PCA encuentra esos nuevos ejes:
- el **componente 1** es la dirección con **más variación** de los datos,
- el **componente 2** es la siguiente, **perpendicular** a la primera, y así sucesivamente.

Cada componente es una **mezcla** de las variables originales, y viene con un porcentaje de
**varianza explicada** (cuánta información conserva). **Siempre se escalan antes** las variables.

**Ejemplo resuelto:** PCA con todos los componentes y su varianza explicada.

In [ ]:
pca_completo = PCA(random_state=SEMILLA).fit(X)
varianza = pca_completo.explained_variance_ratio_
for i, v in enumerate(varianza, start=1):
    print(f"Componente {i}: {v:.1%}")

### Ejercicio 1
Calcula en `varianza_acum` la varianza explicada **acumulada** (usa `np.cumsum` sobre `varianza`)
y en `n_comp_90` el **número mínimo de componentes** que explican al menos el 90 % de la varianza.

In [ ]:
# TODO: np.cumsum da la acumulada; para el mínimo de componentes, busca la primera posición que llega a 0,90 y recuerda que se cuenta desde 1
...

assert abs(varianza_acum[-1] - 1) < 1e-9 and len(varianza_acum) == 7, "varianza_acum debe acabar en 1 y tener 7 valores"
assert n_comp_90 == 5, "Revisa: busca el primer componente donde la acumulada llega a 0.90 y cuéntalo desde 1"
print("Ejercicio 1 correcto ✔ — con", n_comp_90, "componentes conservamos el 90 % de la información")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].bar(range(1, 8), varianza)
ax[0].set_title("Varianza explicada por componente")
ax[0].set_xlabel("Componente"); ax[0].set_ylabel("Proporción")
ax[1].plot(range(1, 8), varianza_acum, marker="o")
ax[1].axhline(0.9, color="tab:red", linestyle="--", label="90 %")
ax[1].set_title("Varianza explicada acumulada")
ax[1].set_xlabel("Número de componentes"); ax[1].legend()
plt.tight_layout(); plt.show()

## 2. Ver a los huéspedes en 2D

**Ejemplo resuelto:** proyectamos a 2 componentes.

In [ ]:
pca2 = PCA(n_components=2, random_state=SEMILLA)
Z2 = pca2.fit_transform(X)
print("Forma:", Z2.shape, "· información conservada:", f"{pca2.explained_variance_ratio_.sum():.1%}")

### Ejercicio 2
Dibuja un diagrama de dispersión de `Z2` (columna 0 en el eje X, columna 1 en el eje Y) coloreando
cada punto según la variable `grupo` (el segmento de k-means). Guarda la figura en `fig2d`.
Pon título y nombres de ejes con el porcentaje de varianza de cada componente.

In [ ]:
# TODO: Crea la figura con plt.subplots y usa ax.scatter con dos columnas de Z2; el argumento c sirve para colorear por grupo
...

assert len(fig2d.axes[0].collections) >= 1, "fig2d debe contener un scatter"
assert fig2d.axes[0].get_title() != "", "Añade un título al gráfico"
plt.show()

Los segmentos se ven **separados** en el plano: PCA nos permite *comprobar a ojo* que el clustering
tiene sentido. Ojo: 2 componentes conservan un ~69 % de la información; lo que ves es una
**sombra** de los datos, no toda la verdad.

## 3. ¿Qué significa cada componente? Las cargas

**Idea.** Un componente es una receta: `0,45·edad + 0,33·gasto − 0,44·%actividades ...`. Los
pesos se llaman **cargas**. Mirar qué variables pesan más (y con qué signo) permite ponerle nombre.
*El signo global de un componente es arbitrario (puede salir invertido), lo que importa es qué
variables van juntas y cuáles en sentido contrario.*

**Ejemplo resuelto:**

In [ ]:
cargas = pd.DataFrame(pca2.components_.T, index=variables, columns=["PC1", "PC2"]).round(2)
cargas

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
cargas.plot.barh(ax=ax)
ax.set_title("Cargas de las variables en los dos primeros componentes")
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout(); plt.show()

### Ejercicio 3
Guarda en `opuestas_a_edad` la lista de variables cuya carga en `PC1` tiene **signo contrario** a la
de `edad` (es decir, cuando la edad sube en el componente 1, ellas bajan). Usa el DataFrame `cargas`.

In [ ]:
# TODO: Compara np.sign(cargas["PC1"]) con np.sign(cargas.loc["edad", "PC1"])
...

assert set(opuestas_a_edad) == {"reservas_anio", "pct_gasto_actividades"}, "Deberían salir 2 variables; revisa el signo de la carga de PC1"
print("Ejercicio 3 correcto ✔ — el componente 1 opone «mayor, planificador, estancias largas» a «activo y viajero frecuente»")

## 4. PCA como paso previo al clustering

**Ejemplo resuelto:** k-means con 4 grupos sobre distintos números de componentes.
*Cuidado:* la silueta calculada en espacios de distinta dimensión **no es comparable** directamente;
aquí solo comprobamos si los grupos son estables.

In [ ]:
from sklearn.metrics import adjusted_rand_score
Zcompleto = PCA(random_state=SEMILLA).fit_transform(X)
for n in (2, 3, 5, 7):
    et = KMeans(n_clusters=4, n_init=10, random_state=SEMILLA).fit_predict(Zcompleto[:, :n])
    print(f"{n} componentes: coincidencia con el clustering original (ARI) = {adjusted_rand_score(grupo, et):.2f}")

### Ejercicio 4
Entrena `PCA(n_components=n_comp_90)` sobre `X`, guarda el resultado en `Z90` y agrupa `Z90` con
k-means (`k=4`, `n_init=10`, `random_state=SEMILLA`). Guarda las etiquetas en `grupo_pca90` y
calcula en `ari_pca90` su coincidencia (ARI) con `grupo`.

In [ ]:
# TODO: PCA con n_components fijo y fit_transform; la coincidencia entre dos agrupaciones se mide con adjusted_rand_score
...

assert Z90.shape == (1200, n_comp_90), "Z90 debe tener n_comp_90 columnas"
assert ari_pca90 > 0.8, "Con el 90 % de la varianza los grupos deberían ser casi los mismos (ARI > 0,8)"
print(f"Ejercicio 4 correcto ✔ — ARI = {ari_pca90:.2f}")

## Una mención a t-SNE
**t-SNE** (y su primo UMAP) es otra técnica para dibujar datos en 2D. A diferencia de PCA,
es **no lineal** y muy buena mostrando grupos, pero: (1) las **distancias entre grupos no significan
nada**, (2) el resultado cambia según los parámetros y la semilla y (3) no tiene "cargas" que
interpretar. Úsalo para **explorar**, no para **demostrar**.

In [ ]:
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, perplexity=30, random_state=SEMILLA, init="pca").fit_transform(X)
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(tsne[:, 0], tsne[:, 1], c=grupo, cmap="tab10", s=8, alpha=0.7)
ax.set_title("t-SNE de los huéspedes (solo para explorar)")
ax.set_xlabel("dimensión 1 (sin unidades)"); ax.set_ylabel("dimensión 2 (sin unidades)")
plt.show()

## Mini-reto integrador
Un compañero te dice: *"con 2 componentes ya vale, quita los demás del modelo"*. Redacta en 3-4 líneas
(en una celda de texto) si estás de acuerdo para **(a)** dibujar los huéspedes y **(b)** entrenar un
modelo predictivo que necesite precisión. Apóyate en `varianza_acum`.

## Preguntas de reflexión
1. ¿Por qué hay que estandarizar antes de PCA? ¿Qué componente dominaría si no lo hicieras (gasto en euros)?
2. Si el componente 1 explica el 43 %, ¿qué significa que el 57 % restante quede "fuera" del gráfico?
3. ¿Es PCA una técnica supervisada? ¿Puede ayudar a un modelo de cancelaciones? ¿Qué se pierde en interpretabilidad?

## Qué has aprendido
- PCA crea ejes nuevos ordenados por la información que conservan.
- La varianza acumulada guía cuántos componentes quedarse.
- Las cargas permiten interpretar los ejes; el signo es arbitrario.
- t-SNE es para explorar visualmente, no para sacar conclusiones cuantitativas.